# Stage 4: Exploratory Analysis - SupplyChain Manufacturing
**Team:** The Leftside Undergrads
**Course:** ISM 6562 - Big Data for Business Applications

This notebook contains ad-hoc analysis and visualizations built on top of the curated and analytics outputs from Stages 1-3.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import pandas as pd

spark = SparkSession.builder \
    .appName('SupplyChain-Exploration') \
    .master('spark://spark-master:7077') \
    .getOrCreate()

BASE = 'hdfs://namenode:9000/data/supplychain'
ANALYTICS = {
    'defect_analysis':    f'{BASE}/analytics/defect_analysis',
    'inventory_risk':     f'{BASE}/analytics/inventory_risk',
    'supplier_scorecard': f'{BASE}/analytics/supplier_scorecard',
    'equipment':          f'{BASE}/analytics/equipment_monitoring',
}
print('Spark session ready.')

## 1. Defect Rate by Factory and Shift

In [ ]:
defects = spark.read.parquet(ANALYTICS['defect_analysis'] + '/defect_by_line_shift')
defects.show(20, truncate=False)

df = defects.toPandas()
pivot = df.pivot_table(index='factory_id', columns='shift', values='defect_rate_pct', aggfunc='mean')
pivot.plot(kind='bar', figsize=(12, 5), title='Average Defect Rate by Factory and Shift')
plt.ylabel('Defect Rate (%)')
plt.xlabel('Factory')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2. Top Defect Codes by Frequency

In [ ]:
defect_codes = spark.read.parquet(ANALYTICS['defect_analysis'] + '/defect_code_summary')
defect_codes.show(20, truncate=False)

df = defect_codes.toPandas().head(10)
plt.figure(figsize=(12, 5))
plt.bar(df['defect_code'], df['total_defects'], color='tomato')
plt.title('Top 10 Defect Codes by Total Frequency')
plt.xlabel('Defect Code')
plt.ylabel('Total Defects')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Inventory Risk - Products Below Reorder Point

In [ ]:
inventory = spark.read.parquet(ANALYTICS['inventory_risk'])
at_risk = inventory.filter(F.col('at_risk') == True)
print(f'Products at risk of stockout: {at_risk.count()}')
at_risk.orderBy(F.asc('days_to_stockout')).show(20, truncate=False)

df = at_risk.toPandas().head(15)
plt.figure(figsize=(12, 5))
plt.barh(df['product_id'], df['days_to_stockout'], color='orange')
plt.title('At-Risk Products - Days Until Stockout')
plt.xlabel('Days to Stockout')
plt.ylabel('Product ID')
plt.tight_layout()
plt.show()

## 4. Supplier Scorecard

In [ ]:
scorecard = spark.read.parquet(ANALYTICS['supplier_scorecard'])
scorecard.show(20, truncate=False)

df = scorecard.toPandas()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(df['supplier_name'], df['avg_on_time_delivery_rate'], color='steelblue')
axes[0].axhline(0.90, color='red', linestyle='--', label='90% target')
axes[0].set_title('On-Time Delivery Rate by Supplier')
axes[0].set_xlabel('Supplier')
axes[0].set_ylabel('Rate')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()
axes[1].bar(df['supplier_name'], df['avg_supplier_defect_rate'], color='tomato')
axes[1].axhline(0.05, color='red', linestyle='--', label='5% threshold')
axes[1].set_title('Defect Rate by Supplier')
axes[1].set_xlabel('Supplier')
axes[1].set_ylabel('Rate')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. Supplier Defect Rate vs Production Defect Rate

In [ ]:
supplier_vs_prod = spark.read.parquet(ANALYTICS['supplier_scorecard'] + '/supplier_vs_production_defects')
supplier_vs_prod.show(20, truncate=False)

df = supplier_vs_prod.toPandas()
plt.figure(figsize=(8, 5))
plt.scatter(df['avg_supplier_defect_rate'], df['production_defect_rate_pct'], color='purple', alpha=0.7)
plt.title('Supplier Defect Rate vs Production Defect Rate')
plt.xlabel('Avg Supplier Defect Rate')
plt.ylabel('Production Defect Rate (%)')
plt.tight_layout()
plt.show()

## 6. Summary of Key Findings

In [ ]:
print('=== KEY FINDINGS ===')
print(f'Total at-risk products: {at_risk.count()}')
print('Exploration complete.')